In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
import glob
import os
import regex as re
import csv

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

# Manaus Population Correction

This notebook corrects the scale mismatch between the population and observed cases.

So far, the project has considered analyzing rural cases in the municipality of Manaus. Given that there is no data based only on this zone, we use a formula derived from the Trajetorias Project (https://www.nature.com/articles/s41597-023-01962-1):

$$\text{Incidence}(d,m,z,t_1,t_2)=\dfrac{\text{Cases}(d,m,z,t_1,t_2)}{\text{Pop}(m,z,(t_1+t_2)/2)\times 5\text{ years}}\times 10^5$$

The parameters are:
<ol>
  <p>$d$: disease (Chagas, CL, VL, Dengue, Falciparum, Vivax, Vivax+Falciparum)</p>
  <p>$m$: municipality</p>
  <p>$z$: zone (rural, urban or total)</p>
  <p>$[t_1,t_2]$: [2004,2008] or [2015, 2019]</p>
</ol>

<!-- ## Problem
- Model was using rural population (~8,558)
- Cases represent ALL of Manaus (~2M population)
- This caused inconsistency in model-data comparison

## Solution Options
1. **Option A**: Scale observed cases to rural level (multiply by 0.0037)
2. **Option B**: Use full municipal population in model (~2M)

We implement both approaches for comparison. -->

## Load Data

In [5]:
DATA_DIR = '../data'

climate_data = pd.read_csv(DATA_DIR + '/climate_api_data_2016_2024.csv')
cases_data = pd.read_csv(DATA_DIR + '/sivep_notification_data/treated_malaria_notification_data/cumulative_manaus_cases_2016_2023.csv')
pop_data = pd.read_csv(DATA_DIR + '/ibge_manaus_population_data_2016_2024.csv')
defor_data = pd.read_csv(DATA_DIR + '/deter_notification_data/treated_deter_deforestation_data_2016_2024.csv')
fires_data = pd.read_csv(DATA_DIR + '/inpe_fire_counts_data_2016_2024.csv')

climate_data['date'] = pd.to_datetime(climate_data['date'])
cases_data['date'] = pd.to_datetime(cases_data['date'])
pop_data['date'] = pd.to_datetime(pop_data['index'])
defor_data['date'] = pd.to_datetime(defor_data['date'])
fires_data['date'] = pd.to_datetime(fires_data['date'])

# Filter to 2017-2023
start_date = pd.to_datetime('2017-01-01')
end_date = pd.to_datetime('2023-12-31')
climate_data = climate_data[(climate_data['date'] >= start_date) & (climate_data['date'] <= end_date)].reset_index(drop=True)
cases_data = cases_data[(cases_data['date'] >= start_date) & (cases_data['date'] <= end_date)].reset_index(drop=True)
pop_data = pop_data[(pop_data['date'] >= start_date) & (pop_data['date'] <= end_date)].reset_index(drop=True)
defor_data = defor_data[(defor_data['date'] >= start_date) & (defor_data['date'] <= end_date)].reset_index(drop=True)
fires_data = fires_data[(fires_data['date'] >= start_date) & (fires_data['date'] <= end_date)].reset_index(drop=True)

In [6]:
with open("../data/TRAJETORIAS_DATASET_Epidemiological_dimension_indicators.csv", encoding="utf-8") as f:
    lines = f.readlines()

clean_lines = [line.replace('"', '') for line in lines]

trajetorias_epidem_df = pd.DataFrame([line.strip().split(",") for line in clean_lines])
trajetorias_epidem_df.columns = trajetorias_epidem_df.iloc[0]
trajetorias_epidem_df = trajetorias_epidem_df[1:].reset_index(drop=True)
trajetorias_epidem_df.columns = trajetorias_epidem_df.columns.str.replace('\ufeff', '')

with open("../data/TRAJETORIAS_DATASET_Population_indicators.csv", encoding="utf-8") as f:
    lines = f.readlines()

clean_lines = [line.replace('"', '') for line in lines]

trajetorias_pop_df = pd.DataFrame([line.strip().split(",") for line in clean_lines])
trajetorias_pop_df.columns = trajetorias_pop_df.iloc[0]
trajetorias_pop_df = trajetorias_pop_df[1:].reset_index(drop=True)
trajetorias_pop_df.columns = trajetorias_pop_df.columns.str.replace('\ufeff', '')

manaus_epidem_df = trajetorias_epidem_df[trajetorias_epidem_df['municipality']=='Manaus']
manaus_epidem_df = manaus_epidem_df.reset_index(drop=True)
manaus_vivax_df = manaus_epidem_df[manaus_epidem_df['disease']=='Vivax']
manaus_vivax_df = manaus_vivax_df.reset_index(drop=True)
manaus_vivax_df = manaus_vivax_df.drop(columns=['state_abbrev', 'state', 'municipality', 'geocode', 'disease'])

cols = ['cases', 'inc']

manaus_vivax_df[cols] = manaus_vivax_df[cols].apply(pd.to_numeric, errors='coerce')

manaus_pop_df = trajetorias_pop_df[trajetorias_pop_df['municipality']=='Manaus']
manaus_pop_df = manaus_pop_df.reset_index(drop=True)
manaus_rural_tot_pop_df = manaus_pop_df.drop(columns=['state_abbrev', 'state', 'municipality', 'geocode', 'prop_urb2000', 'prop_urb2010'])

cols = ['urb2000', 'rur2000', 'tot2000', 'prop_rur2000',
    'urb2010', 'rur2010', 'tot2010', 'prop_rur2010',
    'pop_estimated2006', 'pop_estimated2017',
    'urb2006e', 'rur2006e', 'urb2017e', 'rur2017e']

manaus_rural_tot_pop_df[cols] = manaus_rural_tot_pop_df[cols].apply(pd.to_numeric, errors='coerce')

In [7]:
manaus_vivax_df

,period,zone,cases,inc
0,2004-2008,rural,78745,184030.772087
1,2015-2019,rural,25859,60433.700367
2,2004-2008,urban,183519,2184.793966
3,2015-2019,urban,36387,433.187289
4,2004-2008,total,262264,3106.429047
5,2015-2019,total,62246,584.397051


In [8]:
manaus_rural_tot_pop_df

,urb2000,rur2000,tot2000,prop_rur2000,urb2010,rur2010,tot2010,prop_rur2010,pop_estimated2006,pop_estimated2017,urb2006e,rur2006e,urb2017e,rur2017e
0,1396768,9067,1405835,0.00645,1792881,9133,1802014,0.005068,1688524,2130264,1.679966e+06,8557.807926,2.119467e+06,10796.642597


#### It was noted that the proportions of rural to total population was the same in 2006 and 2017. Checking below, it can be seen that both have the same ratio as 2010:

In [9]:
print(f"Manaus 2000 rural population proportion: {manaus_rural_tot_pop_df['rur2000']/manaus_rural_tot_pop_df['tot2000']}")
print(f"Manaus 2010 rural population proportion: {manaus_rural_tot_pop_df['rur2010']/manaus_rural_tot_pop_df['tot2010']}")
print("--------------------------------------")
print(f"Manaus 2006 rural population proportion estimate: {manaus_rural_tot_pop_df['rur2006e']/manaus_rural_tot_pop_df['pop_estimated2006']}")
print(f"Manaus 2017 rural population proportion estimate: {manaus_rural_tot_pop_df['rur2017e']/manaus_rural_tot_pop_df['pop_estimated2017']}")

Manaus 2000 rural population proportion: 0    0.00645
dtype: float64
Manaus 2010 rural population proportion: 0    0.005068
dtype: float64
--------------------------------------
Manaus 2006 rural population proportion estimate: 0    0.005068
dtype: float64
Manaus 2017 rural population proportion estimate: 0    0.005068
dtype: float64


#### From this, we can see that the estimated rural population does nat change it's proprtion, which may not be appropriate, given the growth of the municipality and especially the process of urbanization. With this in mind, we propose an estimation of the population ratio using the change from 2000 to 2010: